# 用 PROC INBREED 量化电力变压器机型群的共享设计传承

## 执行摘要

某电网运营商变电站变压器按连续的设计世代研制，每一代新机型都衍生自两款前代设计。本笔记本将这种工程谱系视为一份"家系"（pedigree），并使用 **PROC INBREED** 计算近交系数与加性关系（协方差）系数，量化任意两款资产之间共享设计传承的程度。

家系结构经过精心设计以便于解读：当前机队中的四款机型里，有两款衍生自一对**同胞**前代设计，因而携带集中的设计传承；另外两款则来自互不重叠的谱系。该过程精确地还原了这一结构。两款"同胞衍生"机型（`G3_T1`、`G3_T3`）各自的近交系数为 **F = 0.25**；另外两款（`G3_T2`、`G3_T4`）为 **F = 0**。整个机队的平均近交系数为 **0.0417**。查看加性关系矩阵可见，当前机队中关系最疏远的一对是 **`G3_T1` 与 `G3_T3`（关系系数 = 0）**——二者不共享任何祖先设计，构成最强的冗余（N-1）配对，这一点尤为重要，因为 `G3_T1` 本身正是近交程度最高的设计之一。

## 数据来源

| 数据集 | 行数 | 关键变量 | 描述 |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`、`ID`、`Parent1`、`Parent2`、`Sex` | 一份小型、确定性的变电站变压器设计家系，涵盖三个互不重叠的设计世代（4款奠基平台、4款第二代衍生机型、4款当前机队机型）。每款非奠基设计都记录了其衍生自的两款不同前代设计（`Parent1`、`Parent2`）。`Sex` 标记主导设计角色（M = 机械/铁芯谱系，F = 电气/绕组谱系）。该家系为手工指定——而非随机生成——因此亲缘关系结构预先已知，可用于核对该过程的输出结果。 |

# 用 PROC INBREED 量化电力变压器机队的共享设计传承

## 电力公司为何关心"近交"问题

一家输配电运营商要维护数以百计的变电站电力变压器。为控制工程成本与验证工作量，制造商很少从零开始设计每一台变压器——每一款新机型都会从一到两款前代机型**继承**核心几何结构、绕组拓扑、绝缘系统和套管设计。经过多个采购周期，这就形成了一份真实的*工程家系*：一份设计谱系。

这种共享传承是一种隐藏的可靠性风险。如果两台保护同一负荷的变压器（一对冗余的 **N-1** 组合）衍生自同一祖先设计，那么一个潜在的设计缺陷——一种共振模式、一种绝缘老化机制、一条套管闪络路径——很可能同时存在于**两台**设备中。单一根本原因随后就可能同时使这对冗余设备失效：这就是*共模失效*。

**PROC INBREED** 正是为量化这种共享祖源而构建的。尽管它起源于动植物育种领域，但其数学原理——赖特（Wright）的加性关系递推——与具体领域无关：它衡量的是两个个体通过共同祖先所共享的设计传承比例。我们将遗传学词汇映射到资产工程领域：

| INBREED 概念 | 电力公司的解读 |
|---|---|
| 个体 | 一款变压器设计/机型 |
| 亲本（父本/母本） | 其衍生自的一款前代设计 |
| 世代（CLASS） | 一个采购/设计周期 |
| 近交系数 *F* | 一款设计内部自相似传承的程度 |
| 协方差/关系系数 | 两款设计之间的共享设计传承 |
| 关系最疏远的一对 | N-1 冗余韧性的最佳配对 |

## 步骤 1 — 指定设计家系

我们直接录入 `DesignLineage`，使亲缘关系结构一目了然。共有三个**互不重叠的设计世代**：

- **第一代** — 四款奠基平台设计（`G1_P1`-`G1_P4`），无前代设计（亲本为空）。
- **第二代** — 四款衍生设计（`G2_D1`-`G2_D4`），每款均由两款**不同的**第一代平台设计工程衍生而来。`G2_D1` 与 `G2_D2` 为*全同胞*（均来自 `G1_P1` x `G1_P2`）；`G2_D3` 与 `G2_D4` 为全同胞（均来自 `G1_P3` x `G1_P4`）。
- **第三代** — 四款当前机队机型（`G3_T1`-`G3_T4`）。`G3_T1` 由同胞对 `G2_D1` x `G2_D2` 构建而成，`G3_T3` 由同胞对 `G2_D3` x `G2_D4` 构建而成；因此这两款设计集中了传承。`G3_T2` 与 `G3_T4` 则各自跨越了两条互不重叠的谱系。

`ID`、`Parent1` 与 `Parent2` 三列承载家系信息；`Sex` 记录主导该设计的工程谱系。随后的一小段 DATA 步会将占位符 `.` 清空，使奠基平台的亲本字段为空，这是 PROC INBREED 所期望的格式。

In [1]:
数据 DesignLineage;
   长度 ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   输入 Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
运行;

/* 奠基平台没有前代设计：将占位符点号清空 */
数据 DesignLineage;
   设置 DesignLineage;
   如果 Parent1 = '.' 那么 Parent1 = ' ';
   如果 Parent2 = '.' 那么 Parent2 = ' ';
运行;

标题 "变压器设计家系";
过程 打印 数据=DesignLineage noobs;
   变量 Generation ID Parent1 Parent2 Sex;
   标签 Generation="设计世代" ID="设计标识" Parent1="第一亲本设计" Parent2="第二亲本设计" Sex="主导谱系";
运行;

                                                        变压器设计家系                                                         


        设计世代          设计标识              第一亲本设计              第二亲本设计          主导谱系
------------  ------------  ------------------  ------------------  ------------
           1  G1_P1                                                 M
           1  G1_P2                                                 M
           1  G1_P3                                                 M
           1  G1_P4                                                 F
           2  G2_D1         G1_P1               G1_P2               M
           2  G2_D2         G1_P1               G1_P2               F
           2  G2_D3         G1_P3               G1_P4               F
           2  G2_D4         G1_P3               G1_P4               M
           3  G3_T1         G2_D1               G2_D2               M
           3  G3_T2         G2_D1               G2_D3               F
           3  G


NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to 变压器设计家系.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## 步骤 2 — 按设计世代计算近交系数

我们在 `CLASS` 语句中指定 `Generation`，以**多世代模式**运行 PROC INBREED，从而按世代打印家系汇总。`VAR` 语句按要求的顺序提供三个家系列：个体、第一前代、第二前代。

- **IND** 选项打印每款设计的近交系数 *F*——衡量其自身传承集中程度的指标。由两款关系密切的前代设计构建而成的设计，其 *F* 值较高。
- **MATRIX** 选项打印完整的加性关系矩阵，便于我们直接读取两两之间的传承关系。
- **AVERAGE** 选项追加机队整体的平均近交系数——一个单一的设计多样性汇总指标。

数值接近 0 表示谱系相互独立；随着一款设计的两个前代关系越发密切，*F* 值随之升高。

In [2]:
标题 "按设计世代计算的近交系数";
过程 inbreed 数据=DesignLineage ind average MATRIX;
   变量 ID Parent1 Parent2;
   分类 Generation;
   标签 ID="设计标识" Parent1="第一亲本设计" Parent2="第二亲本设计" Generation="设计世代";
运行;

                                                      按设计世代计算的近交系数                                                      


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by 设计世代/Class
--------------------------------------
设计世代 1            Members = 4
设计世代 2            Members = 4
设计世代 3            Members = 4

Inbreeding Coefficients (设计世代 1)
--------------------------------------
设计标识                F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (设计世代 2)
--------------------------------------
设计标识                F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (设计世代 3)
-----------------


NOTE: Option TITLE changed to 按设计世代计算的近交系数.
NOTE: PROC INBREED data=DesignLineage



## 步骤 3 — 保存协方差系数供下游风险评分使用

将近交视图替换为 **COVAR** 选项后，报告的是同一组关系的**协方差（加性关系）系数**形式——这正是资产管理团队用于构建冗余风险评分所需要的形式。**OUTCOV=** 选项将完整矩阵保存为一个数据集（`DesignCovar`），其中 `Col1`-`Col12` 列保存每款设计与其余各设计（按家系顺序排列）之间的关系——可直接与变电站资产分配表关联。

我们打印该输出数据集，使保存下来的矩阵与列表一并可见。

In [3]:
标题 "协方差（关系）系数";
过程 inbreed 数据=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   变量 ID Parent1 Parent2;
   分类 Generation;
   标签 ID="设计标识" Parent1="第一亲本设计" Parent2="第二亲本设计" Generation="设计世代";
运行;

标题 "已保存供风险评分使用的 OUTCOV= 关系矩阵";
过程 打印 数据=DesignCovar noobs;
运行;

                                                       协方差（关系）系数                                                        


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by 设计世代/Class
--------------------------------------
设计世代 1            Members = 4
设计世代 2            Members = 4
设计世代 3            Members = 4

Inbreeding Coefficients (设计世代 1)
--------------------------------------
设计标识                F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (设计世代 1)
--------------------------------------------------
设计标识                   G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.0000
G1_P4                0.0000   0.0


NOTE: Option TITLE changed to 协方差（关系）系数.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to 已保存供风险评分使用的 OUTCOV= 关系矩阵.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## 步骤 4 — 冗余（N-1）安装的最疏远配对

保存下来的 `DesignCovar` 矩阵正是冗余研究所需要的。设计 *i* 与设计 *j* 之间的关系位于第 *i* 行、`Col`*j* 列（各列按家系顺序排列，因此 `Col9`-`Col12` 对应四款当前机队机型 `G3_T1`-`G3_T4`）。

我们从 `DesignCovar` 中读取四款当前机队机型对应的行，构成每一对无序的当前机队配对，附加其关系系数，并按关系疏远程度升序排列。目标是选择共享传承**最低**的冗余配对——这类配对能最大程度降低因单一遗传设计缺陷同时使 N-1 保护位置的两侧同时失效的可能性。

In [4]:
/* 取出四款当前机队机型对应的行；Col9..Col12 分别是与
   G3_T1..G3_T4 的关系系数（按家系列顺序）。构建每一对无序组合。 */
数据 Pairs;
   设置 DesignCovar;
   条件 ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   长度 DesignA $12 DesignB $12;
   数组 g3{4} Col9 Col10 Col11 Col12;
   数组 nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   循环 j = 1 到 4;
      DesignB = nm{j};
      如果 DesignA < DesignB 那么 循环;
         Relationship = g3{j};
         输出;
      结束;
   结束;
   保留 DesignA DesignB Relationship;
运行;

过程 排序 数据=Pairs; 按照 Relationship; 运行;

标题 "候选冗余（N-1）配对，按关系疏远程度升序排列";
过程 打印 数据=Pairs noobs;
   变量 DesignA DesignB Relationship;
   标签 DesignA="设计A" DesignB="设计B" Relationship="关系系数";
运行;
标题;

                                                候选冗余（N-1）配对，按关系疏远程度升序排列                                                 


    设计A      设计B          关系系数
-------  -------  ------------
G3_T1    G3_T3               0
G3_T2    G3_T4            0.25
G3_T1    G3_T2           0.375
G3_T1    G3_T4           0.375
G3_T2    G3_T3           0.375
G3_T3    G3_T4           0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to 候选冗余（N-1）配对，按关系疏远程度升序排列.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## 结果解读

**近交系数（步骤 2）。** 所有奠基第一代平台和全部第二代衍生设计均显示 *F* = 0——按构造，它们都没有两款相关的前代设计。有两款第三代机型显现出 *F* = 0.25：`G3_T1`（构建自同胞对 `G2_D1` x `G2_D2`）和 `G3_T3`（构建自同胞对 `G2_D3` x `G2_D4`）。它们的前代设计可追溯到*同一对*奠基平台，因此其传承高度集中。从可靠性角度看，这些是最容易暴露于单一遗传缺陷的设计，值得进行额外的独立验证测试。而跨越两条互不重叠谱系的 `G3_T2` 与 `G3_T4`，其 *F* = 0。

**关系矩阵（步骤 3）。** 非对角元素量化了两两之间的共享传承。两对同胞第二代设计彼此之间的关系系数均为 0.50（`G2_D1`-`G2_D2` 及 `G2_D3`-`G2_D4`），而来自相反谱系的衍生设计则为 0.00。近交的当前机队设计在对角线上呈现出 1.25（= 1 + *F*）的自身关系值。`DesignCovar` 数据集使完整矩阵可用于与变电站资产分配表关联。

**最疏远配对（步骤 4）。** 按关系系数对每一对当前机队机型排序后，**`G3_T1` 与 `G3_T3` 以关系系数 0.00 排在首位**——它们衍生自完全不重叠的祖先平台，不共享任何设计传承。这是最强的冗余配对，且尤为宝贵，因为 `G3_T1` 本身正是两款近交程度最高的设计之一：将其与一个完全不相关的伙伴配对，可对冲其集中的传承风险。次优配对是 `G3_T2` 与 `G3_T4`，关系系数为 0.25；其余配对的关系系数均为 0.375。机队 **0.0417** 的平均近交系数（由步骤 2 中的 AVERAGE 选项打印）概括了整体设计多样性。采购决策应优先为 N-1 位置选择关系最疏远的配对，并随时间推移拓宽祖先设计基础——这在资产工程领域相当于避免遗传瓶颈。

**注意事项。** 本例为示意性的模拟数据；实际生产研究应基于制造商真实的设计修订记录构建家系，并在据此驱动采购决策之前，对照历史共模停运事件验证亲缘关系评分。